### APEX WEALTH DATA PIPELINE


In [ ]:
%pip install requests

In [ ]:
import requests
import pandas as pd
from dotenv import load_dotenv
import os

In [ ]:

load_dotenv()
api_key = os.getenv('API_KEY')

In [ ]:
API_KEY = os.getenv('API_KEY')

In [ ]:
api_key

In [ ]:
symbol = "AAPL"
url =f"https://api.twelvedata.com/time_series?symbol={symbol}&interval=1min&apikey={api_key}&outputsize=20"


In [ ]:
#Get response from the API
response = requests.get(url)
response.status_code

In [ ]:
url =f"https://api.twelvedata.com/time_series?symbol={symbol}&interval=1min&apikey={api_key}&outputsize=20"
params = {
    "symbol": "AAPL",
    "interval": "1day",
    "apikey": API_KEY
}


In [ ]:


resp = requests.get(url, params=params)
print("Status code:", resp.status_code)  # Should be 200

data = resp.json()
print("Keys in JSON:", data.keys())      # Should include 'values'


In [ ]:
#get data in json format
data = response.json()
data


In [ ]:
df = pd.DataFrame(data["values"])
df.head()

In [ ]:
#transform data to dataframe
def transform_data(data):
    #extract values
    time_series = data['values']

    #convert to dataframe
    df = pd.DataFrame(time_series)

    #convert to proper datatypes
    df['datetime'] = pd.to_datetime(df['datetime'])

    df = df.astype({
        'open': 'float',
        'high': 'float',
        'low': 'float',
        'close': 'float',
        'volume': 'int' })
    return df


In [ ]:
time_series = data['values']
time_series

In [ ]:
df = transform_data(data)

In [ ]:
df

### Extract data for multiple symbols

In [ ]:
symbols = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'TSLA']
all_data = []
def fetch_data(symbol, api_key):
    url =f"https://api.twelvedata.com/time_series?symbol={symbol}&interval=1min&apikey={api_key}&outputsize=5"
    response = requests.get(url)
    response.raise_for_status() #raise an error if we get a bad response
    data = response.json()

    if data.get('status') != 'ok':
        raise ValueError(f'Error fetching data for {symbol}: {data.get("message", "Unknown error")}')
    return data

In [ ]:
import time
symbols = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'TSLA', 'NVDA', 'NFLX', 'META',]


for symbol in symbols:
    data = fetch_data(symbol, api_key)
    df = transform_data(data)
    df['symbol'] = symbol
    all_data.append(df)
    time.sleep(8)
all_data



In [ ]:
all_df = pd.DataFrame()
for symbol in symbols:
    if symbol not in data:
        continue
    symbol_data = data[symbol]

    if "values" not in symbol_data:
        continue

    df = transform_data(symbol_data)
    df['symbol'] = symbol
    all_data.append(df)

In [ ]:
all_df

In [ ]:
#concat - joining or stacking all the dataframes together
all_df = pd.concat(all_data, ignore_index=True)

#### Loading

In [ ]:
load_dotenv()
DB_HOST = os.getenv('DB_HOST')
DB_PORT = os.getenv('DB_PORT')
DB_NAME = os.getenv('DB_NAME')
DB_USER = os.getenv('DB_USER')
DB_PASSWORD = os.getenv('DB_PASSWORD')


In [ ]:
import os
from dotenv import load_dotenv

# Load the .env file
load_dotenv(dotenv_path="C:/full/path/to/your/project/.env")  # change this to your actual path

# Get DB_NAME
DB_NAME = os.getenv("DB_NAME")
print("DB_NAME:", DB_NAME)


In [ ]:
#create a database connnection url 

from sqlalchemy import create_engine 
import psycopg2


db_url = f'postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}'
engine = create_engine(db_url)

#load dataframe to postgres database
all_df.to_sql('stockPrices_data', engine, if_exists='append', index=False)

print("Data loaded to database successfully")             

In [ ]:
all_df.to_csv('stock_data.csv', index=False)

In [ ]:
# Pull request test
